In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
DATA_PATH = "../data/processed/fraud_data_initial.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Date,Account_Balance,Device_Type,Location,Merchant_Category,Previous_Fraudulent_Activity,Daily_Transaction_Count,Card_Type,Card_Age,Fraud_Label,Txn_to_Balance_Pct,Amount_Zscore_User,Day,Month,Day_of_Week,Day_Name
0,TXN_33553,USER_1834,39.79,POS,2023-08-14,93213.17,Laptop,Sydney,Travel,0,7,Amex,65,0,0.043,-0.874,14,8,0,Monday
1,TXN_9427,USER_7875,1.19,Bank Transfer,2023-06-07,75725.25,Mobile,New York,Clothing,0,13,Mastercard,186,1,0.002,-1.050,7,6,2,Wednesday
2,TXN_199,USER_2734,28.96,Online,2023-06-20,1588.96,Tablet,Mumbai,Restaurants,0,14,Visa,226,1,1.823,-0.368,20,6,1,Tuesday
3,TXN_12447,USER_2617,254.32,ATM Withdrawal,2023-12-07,76807.20,Tablet,New York,Clothing,0,8,Visa,76,1,0.331,1.000,7,12,3,Thursday
4,TXN_39489,USER_2014,31.28,POS,2023-11-11,92354.66,Mobile,Mumbai,Electronics,1,14,Mastercard,140,1,0.034,-0.517,11,11,5,Saturday


In [3]:
df.shape

(50000, 20)

In [4]:
df["Date"] = pd.to_datetime(df["Date"])

In [5]:
df["Date"].dtype

dtype('<M8[ns]')

In [6]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Day_of_Week"] = df["Date"].dt.dayofweek

In [7]:
df["Is_Weekend"] = (
    df["Day_of_Week"] >= 5
).astype(int)

In [8]:
df[
    [
        "Date",
        "Year",
        "Month",
        "Day",
        "Day_of_Week",
        "Is_Weekend"
    ]
].head()

,Date,Year,Month,Day,Day_of_Week,Is_Weekend
0,2023-08-14,2023,8,14,0,0
1,2023-06-07,2023,6,7,2,0
2,2023-06-20,2023,6,20,1,0
3,2023-12-07,2023,12,7,3,0
4,2023-11-11,2023,11,11,5,1


In [9]:
df = df.drop(columns=["Date"])

In [10]:
df = df.drop(columns=["Transaction_ID"])

In [11]:
df = df.drop(columns=["User_ID"])

In [12]:
df.columns.tolist()

['Transaction_Amount',
 'Transaction_Type',
 'Account_Balance',
 'Device_Type',
 'Location',
 'Merchant_Category',
 'Previous_Fraudulent_Activity',
 'Daily_Transaction_Count',
 'Card_Type',
 'Card_Age',
 'Fraud_Label',
 'Txn_to_Balance_Pct',
 'Amount_Zscore_User',
 'Day',
 'Month',
 'Day_of_Week',
 'Day_Name',
 'Year',
 'Is_Weekend']

In [13]:
X = df.drop(columns=["Fraud_Label"])
y = df["Fraud_Label"]

In [14]:
X.shape

(50000, 18)

In [15]:
y.shape

(50000,)

In [16]:
numeric_features = [
    "Transaction_Amount",
    "Account_Balance",
    "Previous_Fraudulent_Activity",
    "Daily_Transaction_Count",
    "Card_Age",
    "Txn_to_Balance_Pct",
    "Amount_Zscore_User",
    "Year",
    "Month",
    "Day",
    "Day_of_Week",
    "Is_Weekend"
]

In [17]:
categorical_features = [
    "Transaction_Type",
    "Device_Type",
    "Location",
    "Merchant_Category",
    "Card_Type"
]

In [18]:
print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['Transaction_Amount', 'Account_Balance', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Age', 'Txn_to_Balance_Pct', 'Amount_Zscore_User', 'Year', 'Month', 'Day', 'Day_of_Week', 'Is_Weekend']

Categorical features:
['Transaction_Type', 'Device_Type', 'Location', 'Merchant_Category', 'Card_Type']


In [19]:
print(
    "Total features:",
    len(numeric_features) + len(categorical_features)
)

Total features: 17


In [20]:
for column in categorical_features:
    print("\n", column)
    print(df[column].unique())


 Transaction_Type
['POS' 'Bank Transfer' 'Online' 'ATM Withdrawal']

 Device_Type
['Laptop' 'Mobile' 'Tablet']

 Location
['Sydney' 'New York' 'Mumbai' 'Tokyo' 'London']

 Merchant_Category
['Travel' 'Clothing' 'Restaurants' 'Electronics' 'Groceries']

 Card_Type
['Amex' 'Mastercard' 'Visa' 'Discover']


In [21]:
df[numeric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
Transaction_Amount,50000.0,99.411012,98.687292,0.000,28.6775,69.660,138.8525,1174.140
Account_Balance,50000.0,50294.065981,28760.458557,500.480,25355.9950,50384.430,75115.1350,99998.310
Previous_Fraudulent_Activity,50000.0,0.098400,0.297858,0.000,0.0000,0.000,0.0000,1.000
Daily_Transaction_Count,50000.0,7.485240,4.039637,1.000,4.0000,7.000,11.0000,14.000
Card_Age,50000.0,119.999940,68.985817,1.000,60.0000,120.000,180.0000,239.000
Txn_to_Balance_Pct,50000.0,0.530005,1.956765,0.000,0.0600,0.158,0.3810,105.214
Amount_Zscore_User,50000.0,-0.000204,0.907724,-2.299,-0.7070,-0.267,0.6700,4.295
Year,50000.0,2023.000000,0.000000,2023.000,2023.0000,2023.000,2023.0000,2023.000
Month,50000.0,6.527080,3.446364,1.000,4.0000,7.000,10.0000,12.000
Day,50000.0,15.719320,8.804097,1.000,8.0000,16.000,23.0000,31.000


In [22]:
X.isnull().sum()

Transaction_Amount              0
Transaction_Type                0
Account_Balance                 0
Device_Type                     0
Location                        0
Merchant_Category               0
Previous_Fraudulent_Activity    0
Daily_Transaction_Count         0
Card_Type                       0
Card_Age                        0
Txn_to_Balance_Pct              0
Amount_Zscore_User              0
Day                             0
Month                           0
Day_of_Week                     0
Day_Name                        0
Year                            0
Is_Weekend                      0
dtype: int64

In [23]:
y.isnull().sum()

0

In [24]:
np.isinf(X.select_dtypes(include=np.number)).sum()

Transaction_Amount              0
Account_Balance                 0
Previous_Fraudulent_Activity    0
Daily_Transaction_Count         0
Card_Age                        0
Txn_to_Balance_Pct              0
Amount_Zscore_User              0
Day                             0
Month                           0
Day_of_Week                     0
Year                            0
Is_Weekend                      0
dtype: int64

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [26]:
print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (40000, 18)
Testing : (10000, 18)


In [27]:
print("Training fraud rate:")
print(y_train.mean())

print("\nTesting fraud rate:")
print(y_test.mean())

Training fraud rate:
0.32135

Testing fraud rate:
0.3213


In [28]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

In [29]:
categorical_transformer = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [31]:
X_train_processed = preprocessor.fit_transform(X_train)

In [32]:
X_test_processed = preprocessor.transform(X_test)

In [33]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(40000, 33)
(10000, 33)


In [34]:
preprocessor.fit_transform(X_train)

array([[-0.97740299,  1.6035252 , -0.33138546, ...,  0.        ,
         0.        ,  0.        ],
       [-0.2317831 ,  0.69958478,  3.01763387, ...,  0.        ,
         0.        ,  0.        ],
       [-0.91001431, -1.54571173, -0.33138546, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [ 1.74602922,  1.43634876, -0.33138546, ...,  1.        ,
         0.        ,  0.        ],
       [-0.14964367,  0.17572906, -0.33138546, ...,  0.        ,
         0.        ,  0.        ],
       [-0.7113844 , -0.99646076, -0.33138546, ...,  1.        ,
         0.        ,  0.        ]])

In [35]:
preprocessor.transform(X_test)

array([[ 0.34773935, -1.55726953, -0.33138546, ...,  0.        ,
         0.        ,  1.        ],
       [-0.72977232, -0.57523274, -0.33138546, ...,  0.        ,
         0.        ,  0.        ],
       [-0.67380031,  0.52941281, -0.33138546, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-0.66521255, -0.07746869, -0.33138546, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.53911465, -0.45866715, -0.33138546, ...,  0.        ,
         1.        ,  0.        ],
       [-0.90122448,  0.81949897, -0.33138546, ...,  0.        ,
         0.        ,  0.        ]])

In [36]:
print(type(X_train_processed))

<class 'numpy.ndarray'>


In [37]:
import joblib

joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

['../models/preprocessor.pkl']

In [38]:
train_data = X_train.copy()
train_data["Fraud_Label"] = y_train

test_data = X_test.copy()
test_data["Fraud_Label"] = y_test

In [39]:
train_data.to_csv(
    "../data/processed/train_data.csv",
    index=False
)

test_data.to_csv(
    "../data/processed/test_data.csv",
    index=False
)

In [41]:
print(type(X_train_processed))
print(type(X_test_processed))


<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [42]:
from scipy import sparse

X_train_processed = sparse.csr_matrix(X_train_processed)
X_test_processed = sparse.csr_matrix(X_test_processed)

sparse.save_npz(
    "../data/processed/X_train_processed.npz",
    X_train_processed
)

sparse.save_npz(
    "../data/processed/X_test_processed.npz",
    X_test_processed
)

print("Processed training and testing data saved successfully.")

Processed training and testing data saved successfully.


In [43]:
from scipy import sparse

sparse.save_npz(
    "../data/processed/X_train_processed.npz",
    X_train_processed
)

sparse.save_npz(
    "../data/processed/X_test_processed.npz",
    X_test_processed
)

In [44]:
print("========== PREPROCESSING SUMMARY ==========")
print("Original rows:", len(df))
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Original features:", X.shape[1])
print("Processed training features:", X_train_processed.shape[1])
print("Fraud training samples:", y_train.sum())
print("Fraud testing samples:", y_test.sum())
print("Preprocessing completed successfully.")

========== PREPROCESSING SUMMARY ==========
Original rows: 50000
Training rows: 40000
Testing rows: 10000
Original features: 18
Processed training features: 33
Fraud training samples: 12854
Fraud testing samples: 3213
Preprocessing completed successfully.
